In [7]:
import pandas as pd
import numpy as np
import glob
import os
import seaborn as sns
import matplotlib.pyplot as plt

In [8]:
# --- Configuration ---

# The path to the folder the your CSV files.
FOLDER_PATH = '/fast/AG_Kainmueller/vguarin/aggrigator_experiments/output/tables/auroc_gmm/'

# This dictionary maps the aggregator names in the CSV files
AGGREGATOR_NAME_MAPPING = {
    'Mean': 'AVG',
    'Quantile 0.6': 'AQA 0.60',
    'Quantile 0.75': 'AQA 0.75',
    'Quantile 0.9': 'AQA 0.90',
    'Patch 10': 'PLM 10',
    'Patch 20': 'PLM 20',
    'Patch 50': 'PLM 50',
    'Threshold 0.3': 'ATA 0.3',
    'Threshold 0.5': 'ATA 0.5',
    'Threshold 0.7': 'ATA 0.7',
    'Quantile fg. ratio': 'QFR',
    'Imbalance-w. class avg.': 'ICA', # Assuming this mapping
    'Equally-w. class avg.': 'BCA', # Assuming this mapping
    'GMM_pixel': 'GMM-I',           # Assuming this mapping
    'GMM_spatial': 'GMM-S',         # Assuming this mapping
    'GMM': 'GMM-F',                 # Assuming this mapping
}


# --- Data Loading and Processing ---

# Find all relevant CSV files in the specified folder
search_pattern = os.path.join(FOLDER_PATH, '*_auroc_ood_results.csv')
file_paths = glob.glob(search_pattern)

if not file_paths:
    print(f"Error: No CSV files found at '{search_pattern}'. Please check your FOLDER_PATH.")
else:
    print(f"Found {len(file_paths)} CSV files to process.")

all_results = []

for filepath in file_paths:
    # Extract the dataset name from the filename.
    basename = os.path.basename(filepath)
    dataset_name = basename.replace('_auroc_ood_results.csv', '')

    # Read the CSV data
    temp_df = pd.read_csv(filepath)

    # Standardize aggregator names using the mapping
    temp_df['Aggregator'] = temp_df['Aggregator'].map(AGGREGATOR_NAME_MAPPING).fillna(temp_df['Aggregator'])

    # Keep only the necessary columns and set the index
    temp_df = temp_df[['Aggregator', 'AUROC']].set_index('Aggregator')

    # Rename the 'AUROC' column to the dataset name
    temp_df = temp_df.rename(columns={'AUROC': dataset_name})

    all_results.append(temp_df)

# Combine all individual dataset results into a single DataFrame
summary_df = pd.concat(all_results, axis=1)


# --- Calculate Mean Rank ---

# Rank each aggregator within each dataset column (higher AUROC = better rank)
# 'method="min"' gives the same rank to aggregators with the same score
ranks_df = summary_df.rank(ascending=False, method='min')

# Calculate the mean rank across all datasets for each aggregator
summary_df['Mean Rank'] = ranks_df.mean(axis=1)


# --- Final Formatting and Sorting ---

# Sort the entire table by the new 'Mean Rank' column to see the best performers on top
summary_df = summary_df.sort_values(by='Mean Rank', ascending=True)

# Separate data columns from the rank column for styling
dataset_cols = [col for col in summary_df.columns if col != 'Mean Rank']

# Apply heatmap styling similar to the example image
# Green for high values, red for low, white for middle
styled_df = summary_df.style.background_gradient(
    cmap=sns.diverging_palette(10, 130, as_cmap=True), # Red to Green palette
    subset=dataset_cols,
    low=0.3, # Adjust these to control the color intensity
    high=0.7
).format(
    '{:.2f}', # Format AUROC values to 2 decimal places
    subset=dataset_cols
).format(
    '{:.2f}', # Format Rank value to 2 decimal places
    subset=['Mean Rank']
).set_properties(**{'width': '100px'}) # Adjust column width

print("\n--- Styled Summary Table with Mean Rank ---")
# Display the styled DataFrame in the notebook
display(styled_df)


# --- Save the DataFrame for Sharing ---

# You can save the data in several formats.

# a) Save the raw data (without styles) to a CSV file
output_csv_path = 'auroc_summary_with_ranks.csv'
summary_df.to_csv(output_csv_path)
print(f"\nSuccessfully saved data to '{output_csv_path}'")

# c) To save the styled table as an image, you may need to install a library
# pip install dataframe-image
# import dataframe_image as dfi
#
# output_image_path = 'auroc_summary_table.png'
# dfi.export(styled_df, output_image_path)
# print(f"Successfully saved styled table image to '{output_image_path}'")

Found 10 CSV files to process.

--- Styled Summary Table with Mean Rank ---


,instance_lizard_glas_set_pu,fgbg_wormbodies_protists_pu,instance_arctique_nuclei_intensity_pu,semantic_gta_cityscapes_pu,crops_vs_weed_weedsgalore_maize_pu,fgbg_wormbodies_nematodes_pu,fgbg_lidc_malignancy_pu,semantic_lizard_glas_set_pu,fgbg_lidc_texture_pu,semantic_arctique_blood_cells_pu,Mean Rank
Aggregator,,,,,,,,,,,
BCA,0.68,0.94,0.79,0.89,0.59,0.77,0.57,0.68,0.82,0.90,4.90
ICA,0.71,0.94,0.83,0.84,0.57,0.77,0.57,0.59,0.82,0.82,5.60
GMM-F,0.44,1.00,0.86,1.00,0.95,1.00,0.86,0.44,0.77,0.83,5.60
QFR,0.68,0.91,0.87,0.62,0.58,0.68,0.54,0.57,0.89,0.89,6.20
GMM-I,0.47,1.00,0.83,0.73,0.91,0.98,0.86,0.43,0.78,0.78,6.40
GMM-S,0.49,0.90,0.66,1.00,0.85,0.89,0.67,0.41,0.71,0.93,7.10
AQA 0.60,0.76,0.55,0.64,0.64,0.33,0.48,0.95,0.81,0.50,0.74,8.90
PLM 20,0.67,0.84,0.68,0.42,0.56,0.57,0.95,0.73,0.51,0.71,9.00
AVG,0.76,0.56,0.64,0.74,0.33,0.49,0.95,0.79,0.50,0.63,9.10



Successfully saved data to 'auroc_summary_with_ranks.csv'


In [9]:
# --- Configuration ---

# The path to the folder containing the EAURC CSV files.
FOLDER_PATH = '/fast/AG_Kainmueller/vguarin/aggrigator_experiments/output/tables/eaurc_id_ood/'

# This dictionary maps the aggregator names in the CSV files to the
# final names you want in the table.
AGGREGATOR_NAME_MAPPING = {
    'Mean': 'AVG',
    'Quantile 0.6': 'AQA 0.60',
    'Quantile 0.75': 'AQA 0.75',
    'Quantile 0.9': 'AQA 0.90',
    'Patch 10': 'PLM 10',
    'Patch 20': 'PLM 20',
    'Patch 50': 'PLM 50',
    'Threshold 0.3': 'ATA 0.3',
    'Threshold 0.5': 'ATA 0.5',
    'Threshold 0.7': 'ATA 0.7',
    'Quantile fg. ratio': 'QFR',
    'Imbalance-w. class avg.': 'ICA',
    'Equally-w. class avg.': 'BCA',
    'GMM_pixel': 'GMM-I',
    'GMM_spatial': 'GMM-S',
    'GMM': 'GMM-F',
}


# --- Data Loading and Processing ---

# Find all relevant CSV files in the specified folder
search_pattern = os.path.join(FOLDER_PATH, '*_eaurc_id_ood_results.csv')
file_paths = glob.glob(search_pattern)

if not file_paths:
    print(f"Error: No CSV files found at '{search_pattern}'. Please check your FOLDER_PATH.")
else:
    print(f"Found {len(file_paths)} CSV files to process.")

all_results = []

for filepath in file_paths:
    # Extract the dataset name from the filename
    basename = os.path.basename(filepath)
    dataset_name = basename.replace('_eaurc_id_ood_results.csv', '')

    # Read the CSV data
    temp_df = pd.read_csv(filepath)

    # Standardize aggregator names
    temp_df['Aggregator'] = temp_df['Aggregator'].map(AGGREGATOR_NAME_MAPPING).fillna(temp_df['Aggregator'])

    # Keep only the EAURC column
    temp_df = temp_df[['Aggregator', 'EAURC']].set_index('Aggregator')

    # Rename the column to the dataset name
    temp_df = temp_df.rename(columns={'EAURC': dataset_name})

    all_results.append(temp_df)

# Combine all results into a single DataFrame
summary_df_eaurc = pd.concat(all_results, axis=1)


# --- Calculate Mean Rank (where LOWEST is better) ---

# Rank each aggregator within each dataset column.
# *** CRITICAL CHANGE: ascending=True means lower EAURC gets a better rank (Rank 1) ***
ranks_df_eaurc = summary_df_eaurc.rank(ascending=True, method='min')

# Calculate the mean rank across all datasets for each aggregator
summary_df_eaurc['Mean Rank'] = ranks_df_eaurc.mean(axis=1)


# --- Final Formatting and Sorting ---

# Sort the table by the new 'Mean Rank' column (lower is better)
summary_df_eaurc = summary_df_eaurc.sort_values(by='Mean Rank', ascending=True)

# Separate data columns from the rank column for styling
dataset_cols_eaurc = [col for col in summary_df_eaurc.columns if col != 'Mean Rank']

# Apply heatmap styling
# *** CRITICAL CHANGE: The color map is reversed (Green for low values, Red for high) ***
styled_df_eaurc = summary_df_eaurc.style.background_gradient(
    cmap=sns.diverging_palette(130, 10, as_cmap=True), # Green to Red palette
    subset=dataset_cols_eaurc,
    low=0.3,
    high=0.7
).format(
    '{:.2f}', # Format EAURC values
    subset=dataset_cols_eaurc
).format(
    '{:.2f}', # Format Rank value
    subset=['Mean Rank']
).set_properties(**{'width': '100px'})

print("\n--- Styled EAURC Summary Table with Mean Rank (Lowest is Better) ---")
# Display the styled DataFrame in the notebook
display(styled_df_eaurc)


# --- Save the DataFrame for Sharing ---

# Save the raw data (without styles) to a CSV file
output_csv_path_eaurc = 'eaurc_summary_with_ranks.csv'
summary_df_eaurc.to_csv(output_csv_path_eaurc)
print(f"\nSuccessfully saved data to '{output_csv_path_eaurc}'")

# Optional: To save the styled table as an image
# import dataframe_image as dfi
# output_image_path_eaurc = 'eaurc_summary_table.png'
# dfi.export(styled_df_eaurc, output_image_path_eaurc)
# print(f"Successfully saved styled table image to '{output_image_path_eaurc}'")

Found 10 CSV files to process.

--- Styled EAURC Summary Table with Mean Rank (Lowest is Better) ---


,crops_vs_weed_weedsgalore_maize_pu,semantic_lizard_glas_set_pu,fgbg_wormbodies_nematodes_pu,fgbg_lidc_malignancy_pu,instance_lizard_glas_set_pu,fgbg_wormbodies_protists_pu,fgbg_lidc_texture_pu,semantic_gta_cityscapes_pu,instance_arctique_nuclei_intensity_pu,semantic_arctique_blood_cells_pu,Mean Rank
Aggregator,,,,,,,,,,,
QFR,0.16,0.27,0.04,0.05,0.26,0.04,0.09,0.06,0.01,0.03,3.30
GMM-F,0.08,0.21,0.10,0.07,0.16,0.06,0.07,0.05,0.05,0.06,5.50
GMM-I,0.09,0.20,0.09,0.07,0.17,0.06,0.06,0.10,0.06,0.07,6.20
GMM-S,0.12,0.20,0.14,0.07,0.18,0.07,0.08,0.05,0.10,0.05,6.50
BCA,0.20,0.33,0.10,0.06,0.30,0.09,0.13,0.04,0.03,0.02,6.90
ATA 0.3,0.29,0.28,0.03,0.12,0.27,0.09,0.14,0.09,0.02,0.03,7.30
PLM 20,0.29,0.33,0.09,0.10,0.28,0.06,0.15,0.09,0.02,0.03,8.10
ATA 0.5,0.29,0.28,0.03,0.19,0.23,0.08,0.15,0.10,0.02,0.03,8.20
PLM 10,0.30,0.31,0.11,0.09,0.27,0.05,0.14,0.09,0.03,0.05,8.80



Successfully saved data to 'eaurc_summary_with_ranks.csv'
